# 🚀 Colab T4 GPU — Google Flow 360p → 真 1080p 電影級 AI 超解析工作站

**適用對象**：將本機 `flow_clips_to_upscale.zip`（19 支 Google Flow 360p 片段，約 25 MB）上傳至此，利用免費 **T4 GPU (FP16 Tensor Core)** 以 `spandrel` + `RealESRGAN (realesr-general-x4v3 / RealESRGAN_x4plus)` 逐幀 AI 超解析＋去壓縮紋，直接升頻至 **真 1920×1080**，完成後自動打包下載 `flow_clips_1080p.zip`。

### 使用步驟
1. 上方選單：**執行階段 (Runtime) → 變更執行階段類型 → T4 GPU**
2. 依序執行 Cell 1 → Cell 2（上傳 `flow_clips_to_upscale.zip`）→ Cell 3（自動批次 AI 超分並下載成品）
3. 下載 `flow_clips_1080p.zip` 解壓縮覆蓋回 `peace_love_mv/flow_clips/`，執行 `python mv_builder.py --project peace_love_mv --rebuild-clips` 即可得到真 1080p MV！

In [ ]:
# @title 1️⃣ 檢查 T4 GPU 並安裝零衝突 AI 超分引擎 (spandrel + PyTorch FP16)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q spandrel opencv-python-headless tqdm

import os, urllib.request
from pathlib import Path

# 選擇模型：
#   - realesr-general-x4v3：速度最快（19 鏡約 4-6 分鐘），自然去噪不塑膠臉，最推薦！
#   - RealESRGAN_x4plus：細節更銳利，但速度慢約 3 倍（19 鏡約 14-18 分鐘）
MODEL_CHOICE = "realesr-general-x4v3"  # @param ["realesr-general-x4v3", "RealESRGAN_x4plus"]

MODEL_URLS = {
    "realesr-general-x4v3": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth",
    "RealESRGAN_x4plus": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
}

WEIGHTS_DIR = Path("/content/weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHT_PATH = WEIGHTS_DIR / f"{MODEL_CHOICE}.pth"
if not WEIGHT_PATH.exists():
    print(f"下載權重：{MODEL_CHOICE} ...")
    urllib.request.urlretrieve(MODEL_URLS[MODEL_CHOICE], WEIGHT_PATH)
print(f"✅ 模型就緒：{WEIGHT_PATH} ({WEIGHT_PATH.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# @title 2️⃣ 上傳 flow_clips_to_upscale.zip（或直接掛載 Google Drive）
import zipfile
from pathlib import Path
from google.colab import files

IN_DIR = Path("/content/in_clips")
OUT_DIR = Path("/content/out_clips_1080p")
IN_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("請選擇本機的 flow_clips_to_upscale.zip 上傳...")
uploaded = files.upload()
for fname in uploaded.keys():
    if fname.lower().endswith(".zip"):
        with zipfile.ZipFile(fname, "r") as zf:
            zf.extractall(IN_DIR)
    elif fname.lower().endswith(".mp4"):
        Path(fname).rename(IN_DIR / fname)

mp4s = sorted(IN_DIR.rglob("*.mp4"))
print(f"✅ 共偵測到 {len(mp4s)} 支待超分影片：{[p.name for p in mp4s]}")

In [ ]:
# @title 3️⃣ 啟動 T4 GPU FP16 批次 AI 超分 (360p → 1920×1080) 並自動打包下載
import subprocess, time, zipfile
from pathlib import Path
import cv2
import numpy as np
import torch
from spandrel import ModelLoader
from tqdm.auto import tqdm
from google.colab import files

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_FP16 = DEVICE.type == "cuda"
TARGET_W, TARGET_H = 1920, 1080
BATCH_SIZE = 4 if MODEL_CHOICE == "realesr-general-x4v3" else 2

print(f"載入模型 {MODEL_CHOICE} 至 {DEVICE} (FP16={USE_FP16}, batch={BATCH_SIZE})...")
model_desc = ModelLoader().load_from_file(str(WEIGHT_PATH))
model = model_desc.model.to(DEVICE).eval()
if USE_FP16:
    model = model.half()

@torch.inference_mode()
def upscale_batch(frames_bgr: list[np.ndarray]) -> list[np.ndarray]:
    # BGR uint8 -> RGB float [0,1] NCHW
    rgb = np.stack([cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in frames_bgr], axis=0)
    t = torch.from_numpy(rgb).permute(0, 3, 1, 2).to(DEVICE, non_blocking=True)
    t = t.half().div_(255.0) if USE_FP16 else t.float().div_(255.0)
    out = model(t).clamp_(0.0, 1.0)
    out_np = (out.mul_(255.0).byte().permute(0, 2, 3, 1).cpu().numpy())
    res = []
    for img in out_np:
        bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        # 4x 後精準縮放至 1920x1080（比直接放大更銳利、無鋸齒）
        if bgr.shape[1] != TARGET_W or bgr.shape[0] != TARGET_H:
            bgr = cv2.resize(bgr, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)
        res.append(bgr)
    return res

def upscale_video(src_path: Path, dst_path: Path):
    cap = cv2.VideoCapture(str(src_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

    cmd = [
        "ffmpeg", "-y",
        "-f", "rawvideo", "-vcodec", "rawvideo",
        "-s", f"{TARGET_W}x{TARGET_H}", "-pix_fmt", "bgr24", "-r", f"{fps:.3f}",
        "-i", "-",
        "-c:v", "libx264", "-preset", "medium", "-crf", "17", "-pix_fmt", "yuv420p",
        str(dst_path),
    ]
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    buf = []
    pbar = tqdm(total=total_frames, desc=src_path.name, leave=False)
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            buf.append(frame)
            if len(buf) >= BATCH_SIZE:
                for up in upscale_batch(buf):
                    proc.stdin.write(up.tobytes())
                pbar.update(len(buf))
                buf.clear()
        if buf:
            for up in upscale_batch(buf):
                proc.stdin.write(up.tobytes())
            pbar.update(len(buf))
    finally:
        pbar.close()
        cap.release()
        proc.stdin.close()
        proc.wait()

mp4s = sorted(IN_DIR.rglob("*.mp4"))
t0 = time.time()
for idx, mp4 in enumerate(mp4s, 1):
    out_mp4 = OUT_DIR / mp4.name
    t_clip = time.time()
    upscale_video(mp4, out_mp4)
    print(f"[{idx:02d}/{len(mp4s)}] ✅ {mp4.name} -> {out_mp4.stat().st_size/1e6:.1f} MB ({time.time()-t_clip:.1f}s)")

zip_out = Path("/content/flow_clips_1080p.zip")
with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT_DIR.glob("*.mp4")):
        zf.write(p, arcname=p.name)

print(f"\n🎉 全部完成！總耗時 {(time.time()-t0)/60:.1f} 分鐘，產出：{zip_out} ({zip_out.stat().st_size/1e6:.1f} MB)")
files.download(str(zip_out))